# Import TheLook E-Commerce CSV
อ่าน CSV ทั้ง 7 ไฟล์จากโฟลเดอร์ dataset เป็น pandas DataFrame โดยยังไม่ทำความสะอาดหรือแก้ไขไฟล์ต้นฉบับ

เปิด notebook โดยใช้ project เป็น working directory หากยังไม่มี pandas ให้รัน `%pip install pandas` ในเซลล์ใหม่ก่อน

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "code" / "curated_pipeline.py").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open notebook from the project or code directory")
DATASET_DIR = PROJECT_DIR / "dataset"
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(
        f"Dataset folder not found: {DATASET_DIR}. Set the working directory to the project folder."
    )

## โหลดข้อมูล
เก็บรหัสไปรษณีย์, SKU และ session ID เป็นข้อความเพื่อรักษารูปแบบเดิม ส่วน user_id ใช้ Int64 ที่รองรับค่าว่างใน events ได้ วันที่ยังเก็บเป็นข้อความเพื่อแปลงและตรวจสอบในขั้นตอนถัดไป

ไฟล์ events มีประมาณ 2.4 ล้านแถว การโหลดทั้งชุดอาจใช้เวลาสักครู่และใช้ RAM มากกว่าขนาดไฟล์บนดิสก์

In [ ]:
TABLE_NAMES = [
    "distribution_centers",
    "products",
    "users",
    "orders",
    "inventory_items",
    "order_items",
    "events",
]

missing_files = [
    str(DATASET_DIR / f"{name}.csv")
    for name in TABLE_NAMES
    if not (DATASET_DIR / f"{name}.csv").is_file()
]
if missing_files:
    raise FileNotFoundError(f"Missing CSV files: {missing_files}")

column_types = {
    "postal_code": "string",
    "sku": "string",
    "product_sku": "string",
    "session_id": "string",
    "user_id": "Int64",
}

dataframes = {}
for name in TABLE_NAMES:
    dataframes[name] = pd.read_csv(
        DATASET_DIR / f"{name}.csv",
        encoding="utf-8-sig",
        dtype=column_types,
        keep_default_na=False,
        na_values=[""],
        low_memory=False,
    )
    print(f"Loaded {name}: {len(dataframes[name]):,} rows")

# Convenient names for use in later cells; these reference the same DataFrames.
distribution_centers = dataframes["distribution_centers"]
products = dataframes["products"]
users = dataframes["users"]
orders = dataframes["orders"]
inventory_items = dataframes["inventory_items"]
order_items = dataframes["order_items"]
events = dataframes["events"]

In [ ]:
dataset_summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(df),
            "columns": len(df.columns),
        }
        for name, df in dataframes.items()
    ]
)
display(dataset_summary)

## สำรวจข้อมูลแยกตามตาราง
แต่ละส่วนแสดงจำนวนแถวและคอลัมน์ทั้งหมด พร้อมตัวอย่าง 5 แถวแรก โดยตัวอย่างไม่ใช่ข้อมูลทั้งหมดของตาราง

### orders — คำสั่งซื้อ

In [ ]:
print(f"จำนวนแถวทั้งหมด: {orders.shape[0]:,} แถว")
print(f"จำนวนคอลัมน์ทั้งหมด: {orders.shape[1]:,} คอลัมน์")
print("ตัวอย่างข้อมูล 5 แถวแรก")

with pd.option_context("display.max_columns", None):
    display(orders.head(5))

### order_items — รายการสินค้าในคำสั่งซื้อ

In [ ]:
print(f"จำนวนแถวทั้งหมด: {order_items.shape[0]:,} แถว")
print(f"จำนวนคอลัมน์ทั้งหมด: {order_items.shape[1]:,} คอลัมน์")
print("ตัวอย่างข้อมูล 5 แถวแรก")

with pd.option_context("display.max_columns", None):
    display(order_items.head(5))

### products — สินค้า

In [ ]:
print(f"จำนวนแถวทั้งหมด: {products.shape[0]:,} แถว")
print(f"จำนวนคอลัมน์ทั้งหมด: {products.shape[1]:,} คอลัมน์")
print("ตัวอย่างข้อมูล 5 แถวแรก")

with pd.option_context("display.max_columns", None):
    display(products.head(5))

### users — ผู้ใช้

In [ ]:
print(f"จำนวนแถวทั้งหมด: {users.shape[0]:,} แถว")
print(f"จำนวนคอลัมน์ทั้งหมด: {users.shape[1]:,} คอลัมน์")
print("ตัวอย่างข้อมูล 5 แถวแรก")

with pd.option_context("display.max_columns", None):
    display(users.head(5))

### inventory_items — สินค้าคงคลังรายชิ้น

In [ ]:
print(f"จำนวนแถวทั้งหมด: {inventory_items.shape[0]:,} แถว")
print(f"จำนวนคอลัมน์ทั้งหมด: {inventory_items.shape[1]:,} คอลัมน์")
print("ตัวอย่างข้อมูล 5 แถวแรก")

with pd.option_context("display.max_columns", None):
    display(inventory_items.head(5))

### distribution_centers — ศูนย์กระจายสินค้า

In [ ]:
print(f"จำนวนแถวทั้งหมด: {distribution_centers.shape[0]:,} แถว")
print(f"จำนวนคอลัมน์ทั้งหมด: {distribution_centers.shape[1]:,} คอลัมน์")
print("ตัวอย่างข้อมูล 5 แถวแรก")

with pd.option_context("display.max_columns", None):
    display(distribution_centers.head(5))

### events — เหตุการณ์บนเว็บไซต์

In [ ]:
print(f"จำนวนแถวทั้งหมด: {events.shape[0]:,} แถว")
print(f"จำนวนคอลัมน์ทั้งหมด: {events.shape[1]:,} คอลัมน์")
print("ตัวอย่างข้อมูล 5 แถวแรก")

with pd.option_context("display.max_columns", None):
    display(events.head(5))

In [ ]:
column_profiles = []

for table_name, df in dataframes.items():
    profile = pd.DataFrame({
        "table": table_name,
        "column": df.columns,
        "dtype": df.dtypes.astype(str).to_numpy(),
        "null_count": df.isna().sum().to_numpy(),
        "null_percent": (
            df.isna().mean().mul(100).round(2).to_numpy()
        ),
    })
    column_profiles.append(profile)

column_profile = pd.concat(column_profiles, ignore_index=True)

with pd.option_context("display.max_rows", None):
    display(column_profile)

## ตรวจคุณภาพข้อมูลก่อนทำความสะอาด

รันส่วนนี้ตามลำดับหลังโหลดข้อมูล ผลตรวจใช้ DataFrame ที่โหลดไว้และ **ไม่แก้ไขข้อมูลต้นฉบับ**

- `PASS`: ไม่พบปัญหาตามกฎนั้น, `FAIL`: ผิดกฎโครงสร้าง/ชนิด/คีย์, `WARN`: ต้องทบทวนความหมายทางธุรกิจ, `SKIP`: ตรวจไม่ได้เพราะเงื่อนไขก่อนหน้ายังไม่ผ่าน
- จำนวนปัญหาในแต่ละกฎอาจนับแถวเดียวกันซ้ำ จึงห้ามรวมเป็นจำนวนแถวเสียทั้งหมด
- วันคืน วันส่ง วันขาย และ user_id ของ events ว่างได้ตามบริบท ไม่ใช้ dropna() หรือลบแถวอัตโนมัติ
- ตรวจราคาในชนิดที่อ่านมาเพื่อค้นหาค่าผิดปกติเท่านั้น ยังไม่ได้แปลงเป็น Decimal หรือปัดเศษราคา
- ผลนี้ไม่ได้ตรวจคุณภาพ CSV ก่อน pandas แปลงชนิด เช่น หาก read_csv ล้มเหลว ต้องแก้การอ่าน staging ก่อน และไม่ได้ตรวจความยาว UTF-8 bytes หรือทุกกฎใน Data Dictionary

In [ ]:
import numpy as np
from IPython.display import Markdown

# รันเซลล์นี้ใหม่ก่อนรันการตรวจทั้งหมด เพื่อเริ่มผลลัพธ์ของรอบใหม่
dq_results = []
dq_examples = {}
date_cache = {}

PRIMARY_KEYS = {name: "order_id" if name == "orders" else "id" for name in TABLE_NAMES}
ALLOWED_NULLS = {
    "events": {"user_id"},
    "products": {"name", "brand"},
    "inventory_items": {"sold_at", "product_name", "product_brand"},
    "orders": {"returned_at", "shipped_at", "delivered_at"},
    "order_items": {"returned_at", "shipped_at", "delivered_at"},
}

def missing_mask(series):
    # ตรวจเพิ่ม whitespace-only ที่ isna() เพียงอย่างเดียวไม่พบ
    if pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype):
        return series.isna() | series.astype("string").str.strip().eq("").fillna(False)
    return series.isna()

def record_check(table, rule, mask, severity="FAIL", note="", columns=None, eligible=None):
    df = dataframes[table]
    mask = pd.Series(mask, index=df.index).fillna(False).astype(bool)
    count = int(mask.sum())
    dq_results.append({
        "table": table, "rule": rule,
        "status": severity if count else "PASS",
        "issue_rows": count,
        "checked_rows": len(df) if eligible is None else int(eligible),
        "note": note,
    })
    example_key = f"{table}: {rule}"
    # รันซ้ำกฎเดิมแล้วให้ตัวอย่างเป็นของรอบล่าสุด
    dq_examples.pop(example_key, None)
    if count:
        columns = list(dict.fromkeys([PRIMARY_KEYS[table]] + (columns or [])))
        columns = [c for c in columns if c in df.columns]
        dq_examples[example_key] = df.loc[mask, columns].head(5).copy()

def skip_check(table, rule, note):
    dq_results.append({"table": table, "rule": rule, "status": "SKIP",
                       "issue_rows": pd.NA, "checked_rows": 0, "note": note})

def show_checks(start):
    display(pd.DataFrame(dq_results[start:]))

### 1. ชนิดข้อมูลและค่าว่าง แยกตามตาราง

แสดง NULL, ข้อความว่าง/ช่องว่างล้วน และข้อความคล้าย null (`NULL`, `N/A`, `NA`, `NONE`, `NAN`) แยกกัน ข้อความคล้าย null เป็นเพียงคำเตือน ต้องทบทวนก่อนแปลง เพราะอาจเป็นค่าที่มีความหมายจริง

In [ ]:
profile_rows = []
for table_name, df in dataframes.items():
    display(Markdown(f"#### {table_name}"))
    start = len(profile_rows)
    for column in df.columns:
        series = df[column]
        missing = missing_mask(series)
        textual = pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)
        tokens = (series.astype("string").str.strip().str.upper().isin(["NULL", "N/A", "NA", "NONE", "NAN"])
                  if textual else pd.Series(False, index=df.index))
        allowed = column in ALLOWED_NULLS.get(table_name, set())
        profile_rows.append({
            "table": table_name, "column": column, "dtype": str(series.dtype),
            "null_rows": int(series.isna().sum()),
            "blank_text_rows": int((missing & series.notna()).sum()),
            "missing_percent": round(float(missing.mean() * 100), 2) if len(df) else 0,
            "null_allowed": allowed, "null_like_text_rows": int(tokens.sum()),
        })
        if not allowed:
            record_check(table_name, f"required:{column}", missing, columns=[column])
        if textual:
            record_check(table_name, f"null_like_text:{column}", tokens, severity="WARN", columns=[column])
    display(pd.DataFrame(profile_rows[start:]).drop(columns="table"))

quality_column_profile = pd.DataFrame(profile_rows)

### 2. คีย์หลักและคีย์อ้างอิงต้องเป็นจำนวนเต็มบวก

ตรวจคีย์ว่าง คีย์ซ้ำ และรูปแบบตัวเลข การนับ duplicate ใช้ทุกแถวที่อยู่ในกลุ่มซ้ำ (`keep=False`) ไม่ใช่เฉพาะแถวเกินตัวแรก ส่วน SKU และ session_id เป็นข้อความและไม่อยู่ในกฎนี้

In [ ]:
start = len(dq_results)
for table_name, df in dataframes.items():
    key = PRIMARY_KEYS[table_name]
    missing = missing_mask(df[key])
    record_check(table_name, "primary_key_missing", missing, columns=[key])
    record_check(table_name, "primary_key_duplicate", ~missing & df[key].duplicated(keep=False), columns=[key])
    key_columns = [c for c in df.columns if c == "id" or (c.endswith("_id") and c != "session_id")]
    for column in key_columns:
        values = pd.to_numeric(df[column], errors="coerce")
        present = ~missing_mask(df[column])
        invalid = present & (values.isna() | ~np.isfinite(values) | values.le(0) | values.mod(1).ne(0))
        record_check(table_name, f"positive_integer_key:{column}", invalid, columns=[column])
show_checks(start)

### 3. ความสัมพันธ์ระหว่างตารางทั้ง 9 เส้นทาง

ตรวจเฉพาะ FK ที่ไม่ว่าง ส่วนคีย์ว่างที่ห้ามว่างตรวจในข้อ 1 แล้ว การมี NULL ใน events.user_id ไม่ถือว่าเป็น orphan คีย์ที่ชนิดผิดต้องแก้ตามผลข้อ 2 ก่อน จึงไม่ตีความผล FK อย่างเดียวว่าเพียงพอ

In [ ]:
RELATIONSHIPS = [
    ("products", "distribution_center_id", "distribution_centers", "id"),
    ("inventory_items", "product_id", "products", "id"),
    ("inventory_items", "product_distribution_center_id", "distribution_centers", "id"),
    ("orders", "user_id", "users", "id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "user_id", "users", "id"),
    ("order_items", "product_id", "products", "id"),
    ("order_items", "inventory_item_id", "inventory_items", "id"),
    ("events", "user_id", "users", "id"),
]
start = len(dq_results)
for child, foreign_key, parent, parent_key in RELATIONSHIPS:
    child_values = pd.to_numeric(dataframes[child][foreign_key], errors="coerce")
    parent_values = pd.to_numeric(dataframes[parent][parent_key], errors="coerce").dropna()
    present = ~missing_mask(dataframes[child][foreign_key])
    record_check(child, f"foreign_key:{foreign_key}->{parent}.{parent_key}",
                 present & ~child_values.isin(parent_values),
                 columns=[foreign_key], eligible=present.sum())
show_checks(start)

### 4. วันเวลา: รูปแบบและลำดับเหตุการณ์

แปลงลงตัวแปรชั่วคราวเป็น UTC โดยไม่เปลี่ยนคอลัมน์ต้นฉบับ ตรวจ timezone ที่ไม่มีแยกต่างหาก ไม่ใช้วันที่ที่ไม่มี timezone/parse ไม่ได้เปรียบเทียบลำดับ วันเวลาผิดลำดับเป็น WARN เพื่อให้ตรวจความหมายของแหล่งข้อมูลก่อน โดยเฉพาะ created_at ของ order item

In [ ]:
start = len(dq_results)
date_ranges = []
for table_name, df in dataframes.items():
    for column in [c for c in df.columns if c.endswith("_at")]:
        source = df[column].astype("string").str.strip()
        present = ~missing_mask(df[column])
        has_zone = source.str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", case=False, regex=True, na=False)
        parsed = pd.to_datetime(source.where(present), format="mixed", utc=True, errors="coerce")
        record_check(table_name, f"datetime_parse:{column}", present & parsed.isna(), columns=[column], eligible=present.sum())
        record_check(table_name, f"datetime_timezone:{column}", present & ~has_zone, columns=[column], eligible=present.sum())
        usable = parsed.where(has_zone)
        date_cache[(table_name, column)] = usable
        date_ranges.append({"table": table_name, "column": column,
                            "min_utc": usable.min(), "max_utc": usable.max(),
                            "valid_with_timezone": int(usable.notna().sum())})

DATE_PAIRS = {
    "orders": [("created_at", "shipped_at"), ("shipped_at", "delivered_at"), ("delivered_at", "returned_at")],
    "order_items": [("created_at", "shipped_at"), ("shipped_at", "delivered_at"), ("delivered_at", "returned_at")],
    "inventory_items": [("created_at", "sold_at")],
}
for table_name, pairs in DATE_PAIRS.items():
    for earlier, later in pairs:
        a, b = date_cache[(table_name, earlier)], date_cache[(table_name, later)]
        comparable = a.notna() & b.notna()
        record_check(table_name, f"date_order:{later}>={earlier}", comparable & b.lt(a),
                     severity="WARN", columns=[earlier, later], eligible=comparable.sum(),
                     note="ตรวจเฉพาะแถวที่มีวันที่ทั้งคู่และ timezone ถูกต้อง; ทบทวนก่อนแก้ไข")
show_checks(start)
display(pd.DataFrame(date_ranges))

### 5. ราคา จำนวน อายุ และพิกัด

ค่าที่ไม่ใช่ตัวเลขหรือ infinite เป็น FAIL ส่วนราคาติดลบและอายุนอกช่วงที่เสนอเป็น WARN การตรวจช่วงไม่ใช่การยืนยันสกุลเงินหรือความถูกต้องของราคาทางธุรกิจ

In [ ]:
NUMERIC_RULES = {
    "products": [("cost", 0, None, False, "WARN"), ("retail_price", 0, None, False, "WARN")],
    "inventory_items": [("cost", 0, None, False, "WARN"), ("product_retail_price", 0, None, False, "WARN")],
    "order_items": [("sale_price", 0, None, False, "WARN")],
    "orders": [("num_of_item", 1, None, True, "FAIL")],
    "users": [("age", 0, 120, True, "WARN"), ("latitude", -90, 90, False, "FAIL"), ("longitude", -180, 180, False, "FAIL")],
    "distribution_centers": [("latitude", -90, 90, False, "FAIL"), ("longitude", -180, 180, False, "FAIL")],
    "events": [("sequence_number", 1, None, True, "WARN")],
}
start = len(dq_results)
for table_name, rules in NUMERIC_RULES.items():
    for column, minimum, maximum, integer, severity in rules:
        source = dataframes[table_name][column]
        values = pd.to_numeric(source, errors="coerce")
        present = ~missing_mask(source)
        finite = values.notna() & np.isfinite(values)
        record_check(table_name, f"numeric_parse:{column}", present & ~finite, columns=[column], eligible=present.sum())
        outside = values.lt(minimum)
        if maximum is not None:
            outside |= values.gt(maximum)
        if integer:
            outside |= values.mod(1).ne(0)
        record_check(table_name, f"numeric_range:{column}", finite & outside,
                     severity=severity, columns=[column], eligible=finite.sum(),
                     note=f"min={minimum}, max={maximum}, integer={integer}")
show_checks(start)

### 6. ความสอดคล้องของคำสั่งซื้อและสินค้าคงคลัง

ใช้ map จาก parent ที่คีย์ไม่ซ้ำเพื่อไม่ให้ join เพิ่มจำนวนแถว หาก parent key ผิดให้ SKIP อย่างชัดเจน สถานะต่างกันและ inventory item ที่ถูกอ้างซ้ำเป็น WARN เพราะต้องยืนยันกติกาธุรกิจก่อนตัดสินว่าเป็นข้อมูลเสีย

In [ ]:
start = len(dq_results)
item_counts = order_items.groupby("order_id").size()
observed_counts = orders["order_id"].map(item_counts).fillna(0)
declared_counts = pd.to_numeric(orders["num_of_item"], errors="coerce")
record_check("orders", "item_count_matches", declared_counts.ne(observed_counts), columns=["num_of_item"],
             note="เทียบ num_of_item กับจำนวนแถว order_items ต่อ order_id")

def compare_parent(child, foreign_key, parent, parent_key, child_column, parent_column, severity="FAIL"):
    rule = f"parent_consistency:{child_column}->{parent}.{parent_column}"
    parent_df = dataframes[parent]
    if missing_mask(parent_df[parent_key]).any() or parent_df[parent_key].duplicated().any():
        skip_check(child, rule, "parent key ว่าง/ซ้ำ ต้องตรวจแก้ก่อนเปรียบเทียบ")
        return
    child_df = dataframes[child]
    lookup = parent_df.set_index(parent_key)[parent_column]
    matched = child_df[foreign_key].isin(lookup.index)
    parent_values = child_df[foreign_key].map(lookup)
    comparable = matched & ~missing_mask(child_df[child_column]) & ~missing_mask(parent_values)
    if not comparable.any():
        skip_check(child, rule, "ไม่มีแถวที่จับคู่และมีค่าทั้งสองด้านได้ ตรวจผล FK/required fields")
        return
    record_check(child, rule, comparable & child_df[child_column].ne(parent_values),
                 severity=severity, columns=[foreign_key, child_column], eligible=comparable.sum())

compare_parent("order_items", "order_id", "orders", "order_id", "user_id", "user_id")
compare_parent("order_items", "order_id", "orders", "order_id", "status", "status", "WARN")
compare_parent("order_items", "inventory_item_id", "inventory_items", "id", "product_id", "product_id")
record_check("order_items", "inventory_item_reused",
             ~missing_mask(order_items["inventory_item_id"]) & order_items["inventory_item_id"].duplicated(keep=False),
             severity="WARN", columns=["inventory_item_id", "order_id"])

known_statuses = {"Processing", "Shipped", "Complete", "Cancelled", "Returned"}
for table_name in ["orders", "order_items"]:
    df = dataframes[table_name]
    record_check(table_name, "unexpected_status", ~missing_mask(df["status"]) & ~df["status"].isin(known_statuses),
                 severity="WARN", columns=["status"])
show_checks(start)

# ดูค่าว่างตามสถานะเพื่อช่วยกำหนดกฎ ไม่สรุปว่าทุกค่าว่างเป็นปัญหา
for table_name in ["orders", "order_items"]:
    display(Markdown(f"#### {table_name}: จำนวนวันที่ว่างแยกตามสถานะ"))
    df = dataframes[table_name]
    date_columns = ["shipped_at", "delivered_at", "returned_at"]
    missing_dates = pd.DataFrame({c: missing_mask(df[c]) for c in date_columns})
    by_status = missing_dates.groupby(df["status"], dropna=False).sum()
    by_status.insert(0, "rows_in_status", df.groupby("status", dropna=False).size())
    display(by_status)

### 7. สรุปผลและตัวอย่างแถวที่ต้องทบทวน

สรุปจำนวน **กฎ** ที่ผ่าน/ไม่ผ่าน ไม่ใช่จำนวนแถวเสียที่ไม่ซ้ำ หากมี FAIL ควรตรวจตัวอย่างและกำหนดวิธีจัดการก่อนทำ curated ส่วน WARN ต้องพิจารณาความหมายทางธุรกิจ ตารางตัวอย่างแสดงสูงสุด 5 แถวต่อกฎ โดยไม่แสดงข้อมูลส่วนบุคคลที่ไม่เกี่ยวข้อง

In [ ]:
# เก็บเฉพาะผลล่าสุดของแต่ละกฎหากเผลอรันบางเซลล์ซ้ำ
dq_summary = pd.DataFrame(dq_results).drop_duplicates(["table", "rule"], keep="last").reset_index(drop=True)
status_order = ["FAIL", "WARN", "SKIP", "PASS"]
dq_overview = pd.crosstab(dq_summary["table"], dq_summary["status"]).reindex(columns=status_order, fill_value=0)
display(dq_overview.rename_axis(columns="จำนวนกฎ"))

dq_issues = dq_summary.loc[dq_summary["status"] != "PASS"].copy()
if dq_issues.empty:
    print("ไม่พบปัญหาตามกฎที่ตรวจในรอบนี้ ยังไม่ใช่การรับรองคุณภาพทุกด้าน")
else:
    display(dq_issues.sort_values(["table", "status", "rule"]))
    for row in dq_issues.itertuples(index=False):
        key = f"{row.table}: {row.rule}"
        if key in dq_examples:
            display(Markdown(f"#### {row.status} — {key}"))
            display(dq_examples[key])

# เรียกดูผลรายตารางได้ เช่น:
# display(dq_summary.loc[dq_summary["table"] == "orders"])

## วิเคราะห์คำเตือนก่อนกำหนดกฎทำความสะอาด

รันหลังเซลล์โหลด CSV ส่วนนี้คำนวณจาก `users`, `events`, `orders` และ `order_items` โดยตรง จึงไม่จำเป็นต้องรันการตรวจคุณภาพทั้งหมดซ้ำ และไม่แก้ไข DataFrame ต้นฉบับ

วิเคราะห์ข้อความคล้าย NULL ใน city และวันที่ส่งก่อนวันสร้างรายการ แสดงจำนวน ค่าจริง และบริบทที่เกี่ยวข้อง ผลเหล่านี้เป็นหลักฐานเพื่อทบทวน ไม่ได้ยืนยันสาเหตุหรืออนุญาตให้แทนค่าโดยอัตโนมัติ

In [ ]:
from IPython.display import display, Markdown

required_tables = ["users", "events", "orders", "order_items"]
unloaded = [name for name in required_tables if name not in globals()]
if unloaded:
    raise RuntimeError(f"รันเซลล์โหลดข้อมูลก่อน: {unloaded}")

NULL_LIKE_CITY = {"NULL", "N/A", "NA", "NONE", "NAN"}
city_warning_details = {}
city_warning_rows = []

for table_name, df in [("users", users), ("events", events)]:
    city_text = df["city"].astype("string")
    warning_mask = city_text.str.strip().str.upper().isin(NULL_LIKE_CITY)
    columns = [c for c in ["id", "user_id", "city", "state", "country", "postal_code"] if c in df.columns]
    detail = df.loc[warning_mask, columns].copy()
    # repr ช่วยให้เห็นช่องว่างและตัวพิมพ์ของค่าจริงชัดเจน
    detail["city_raw_repr"] = detail["city"].map(repr)
    city_warning_details[table_name] = detail
    city_warning_rows.append({
        "table": table_name,
        "total_rows": len(df),
        "null_like_city_rows": len(detail),
        "percent_of_table": round(100 * len(detail) / len(df), 2) if len(df) else 0,
        "actual_null_city_rows": int(df["city"].isna().sum()),
    })

city_warning_summary = pd.DataFrame(city_warning_rows)
display(city_warning_summary)

### 1. ค่าจริงของ city และพื้นที่ที่พบ

ตารางความถี่นับจากแถวทั้งหมดที่ถูกเตือน ส่วนตัวอย่างจำกัด 10 แถว และบริบทพื้นที่แสดง 20 กลุ่มที่พบบ่อยที่สุด ไม่แสดงชื่อ อีเมล ที่อยู่ หรือ IP ที่ไม่จำเป็นต่อการวิเคราะห์นี้

In [ ]:
for table_name, detail in city_warning_details.items():
    display(Markdown(f"#### {table_name}"))
    if detail.empty:
        print("ไม่พบข้อความ city ที่ตรงกับรายการคำเตือน")
        continue

    print("ค่าจริงและจำนวนครั้งที่พบ")
    display(detail.groupby("city_raw_repr", dropna=False).size().rename("rows").reset_index().sort_values("rows", ascending=False))

    print("บริบทพื้นที่ที่พบมากที่สุด 20 กลุ่ม")
    group_columns = [c for c in ["city_raw_repr", "country", "state", "postal_code"] if c in detail.columns]
    city_context = (detail.groupby(group_columns, dropna=False).size()
                    .rename("rows").reset_index().sort_values("rows", ascending=False))
    display(city_context.head(20))

    print("ตัวอย่างสูงสุด 10 แถว")
    display(detail.head(10))

### 2. เทียบ city ของ events ที่ถูกเตือนกับ city ของผู้ใช้

ช่วยดูว่าเหตุการณ์และผู้ใช้ที่อ้างถึงมีค่าตรงกันหรือไม่ โดยใช้ LEFT JOIN เพื่อรักษา events ที่ไม่มี user_id ตำแหน่งของผู้ใช้อาจต่างจากตำแหน่งขณะเกิดเหตุการณ์ จึงห้ามนำ users.city ไปเติม events.city อัตโนมัติ

In [ ]:
event_city_comparison = None
if users["id"].isna().any() or users["id"].duplicated().any():
    print("SKIP: users.id ว่างหรือซ้ำ จึงยังเทียบกับ events ไม่ได้")
else:
    user_locations = users[["id", "city", "state", "country"]].rename(columns={
        "id": "matched_user_id", "city": "user_city", "state": "user_state", "country": "user_country",
    })
    event_city_comparison = city_warning_details["events"].merge(
        user_locations, how="left", left_on="user_id", right_on="matched_user_id",
        validate="many_to_one", indicator=True,
    )
    event_city_comparison["comparison"] = "user_id_not_found"
    event_city_comparison.loc[event_city_comparison["user_id"].isna(), "comparison"] = "no_user_id"
    matched = event_city_comparison["_merge"].eq("both")
    user_city = event_city_comparison["user_city"].astype("string").str.strip().str.upper()
    event_city = event_city_comparison["city"].astype("string").str.strip().str.upper()
    missing_user_city = user_city.isna() | user_city.eq("").fillna(False)
    event_city_comparison.loc[matched & missing_user_city, "comparison"] = "user_city_missing"
    comparable = matched & ~missing_user_city
    event_city_comparison.loc[comparable, "comparison"] = "different_city_text"
    event_city_comparison.loc[comparable & user_city.eq(event_city).fillna(False), "comparison"] = "same_city_text"
    display(event_city_comparison["comparison"].value_counts(dropna=False).rename_axis("comparison").reset_index(name="rows"))
    display(event_city_comparison[["id", "user_id", "city", "state", "user_city", "user_state", "user_country", "comparison"]].head(10))

### 3. เทียบวันเวลาของ order_items กับ orders

เชื่อมด้วย order_id และตรวจว่า parent ไม่ซ้ำก่อน ใช้วันที่ที่ parse ได้และมี timezone เท่านั้นในการเปรียบเทียบ ค่าช่องว่างหรือ parse ไม่ได้จะไม่ถูกนับว่าเป็นวันที่ผิดลำดับ และแสดงจำนวนแถวที่เปรียบเทียบได้แยกต่างหาก

`item_create_minus_order_create_hours` เป็นชั่วโมงที่วันสร้างรายการห่างจากวันสร้าง order ส่วน `ship_minus_item_create_hours` ติดลบหมายถึงวันส่งก่อนวันสร้างรายการ ไม่ปรับวันที่ทั้งสองให้เท่ากัน

In [ ]:
def parse_aware_utc(series):
    text = series.astype("string").str.strip()
    has_timezone = text.str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", regex=True, case=False, na=False)
    return pd.to_datetime(text.where(has_timezone), format="mixed", utc=True, errors="coerce")

date_comparison = None
date_warning_rows = None
date_warning_metrics = pd.DataFrame()

if orders["order_id"].isna().any() or orders["order_id"].duplicated().any():
    print("SKIP: orders.order_id ว่างหรือซ้ำ ต้องตรวจคีย์ก่อนเปรียบเทียบ")
else:
    item_dates = order_items[["id", "order_id", "status", "created_at", "shipped_at"]].rename(columns={
        "status": "item_status", "created_at": "item_created_at", "shipped_at": "item_shipped_at",
    })
    parent_dates = orders[["order_id", "status", "created_at", "shipped_at"]].rename(columns={
        "status": "order_status", "created_at": "order_created_at", "shipped_at": "order_shipped_at",
    })
    date_comparison = item_dates.merge(parent_dates, on="order_id", how="left", validate="many_to_one", indicator=True)
    for column in ["item_created_at", "item_shipped_at", "order_created_at", "order_shipped_at"]:
        date_comparison[column + "_utc"] = parse_aware_utc(date_comparison[column])

    item_created = date_comparison["item_created_at_utc"]
    item_shipped = date_comparison["item_shipped_at_utc"]
    order_created = date_comparison["order_created_at_utc"]
    order_shipped = date_comparison["order_shipped_at_utc"]
    item_comparable = item_created.notna() & item_shipped.notna()
    date_comparison["item_date_comparable"] = item_comparable
    date_comparison["item_shipped_before_created"] = item_comparable & item_shipped.lt(item_created)
    date_comparison["item_create_minus_order_create_hours"] = (item_created - order_created).dt.total_seconds() / 3600
    date_comparison["ship_minus_item_create_hours"] = (item_shipped - item_created).dt.total_seconds() / 3600
    date_comparison["ship_minus_order_create_hours"] = (item_shipped - order_created).dt.total_seconds() / 3600
    date_warning_rows = date_comparison.loc[date_comparison["item_shipped_before_created"]].copy()

    def date_metric(label, eligible, issue):
        checked = int(eligible.sum())
        count = int((eligible & issue).sum())
        return {"check": label, "checked_item_rows": checked, "issue_item_rows": count,
                "issue_percent": round(100 * count / checked, 2) if checked else pd.NA}

    date_warning_metrics = pd.DataFrame([
        date_metric("item shipped before item created", item_comparable, item_shipped.lt(item_created)),
        date_metric("item created differs from order created", item_created.notna() & order_created.notna(), item_created.ne(order_created)),
        date_metric("item shipped differs from order shipped", item_shipped.notna() & order_shipped.notna(), item_shipped.ne(order_shipped)),
        date_metric("order shipped before order created (นับระดับ item)", order_shipped.notna() & order_created.notna(), order_shipped.lt(order_created)),
    ])
    display(date_warning_metrics)
    print(f"แถว item ที่จับคู่ order ไม่ได้: {date_comparison['_merge'].eq('left_only').sum():,}")
    print(f"แถว item ที่วันที่สร้าง/ส่งไม่ครบหรือไม่ผ่านการ parse พร้อม timezone: {(~item_comparable).sum():,}")
    print(f"จำนวน order ไม่ซ้ำที่มี item ถูกเตือน: {date_warning_rows['order_id'].nunique():,}")

### 4. เจาะกลุ่มวันที่ถูกเตือนและดูตัวอย่าง

แยกตามสถานะ แสดงสถิติระยะห่าง และดูรายการที่วันส่งอยู่ก่อนวันสร้างมากที่สุด การที่วันที่ order ดูสมเหตุสมผลกว่าไม่ได้พิสูจน์ว่าต้องใช้ทับวันที่ item ต้องยืนยันความหมายกับแหล่งข้อมูลก่อน

In [ ]:
if date_comparison is None:
    print("SKIP: ยังไม่มีผลเปรียบเทียบวันที่")
else:
    date_status_summary = date_comparison.groupby("item_status", dropna=False).agg(
        total_item_rows=("id", "size"),
        comparable_item_rows=("item_date_comparable", "sum"),
        warning_item_rows=("item_shipped_before_created", "sum"),
    )
    date_status_summary["warning_percent_of_comparable"] = (
        100 * date_status_summary["warning_item_rows"]
        / date_status_summary["comparable_item_rows"].replace(0, float("nan"))
    ).round(2)
    display(date_status_summary)

    if date_warning_rows.empty:
        print("ไม่พบรายการที่ส่งก่อนวันสร้างในข้อมูลที่เปรียบเทียบได้")
    else:
        print("สถิติระยะห่างเป็นชั่วโมง เฉพาะรายการที่ถูกเตือน")
        delta_columns = ["item_create_minus_order_create_hours", "ship_minus_item_create_hours", "ship_minus_order_create_hours"]
        display(date_warning_rows[delta_columns].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).T)

        print("ตัวอย่าง 10 รายการที่วันส่งก่อนวันสร้างรายการมากที่สุด")
        example_columns = ["id", "order_id", "item_status", "order_status", "item_created_at", "order_created_at",
                           "item_shipped_at", "order_shipped_at"] + delta_columns
        with pd.option_context("display.max_columns", None):
            display(date_warning_rows.sort_values("ship_minus_item_create_hours")[example_columns].head(10))

        warning = date_comparison["item_shipped_before_created"]
        same_ship_comparable = warning & date_comparison["item_shipped_at_utc"].notna() & date_comparison["order_shipped_at_utc"].notna()
        create_comparable = warning & date_comparison["item_created_at_utc"].notna() & date_comparison["order_created_at_utc"].notna()
        within_warning_checks = pd.DataFrame([
            date_metric("วันที่ส่ง item ต่างจาก order ในกลุ่มเตือน", same_ship_comparable,
                        date_comparison["item_shipped_at_utc"].ne(date_comparison["order_shipped_at_utc"])),
            date_metric("วันที่สร้าง item ต่างจาก order ในกลุ่มเตือน", create_comparable,
                        date_comparison["item_created_at_utc"].ne(date_comparison["order_created_at_utc"])),
        ])
        display(within_warning_checks)

### 5. บันทึกสิ่งที่ต้องตัดสินใจก่อน cleaning

| ประเด็น | หลักฐานจากเซลล์ด้านบน | สิ่งที่ต้องยืนยัน |
|---|---|---|
| city เป็นข้อความคล้าย NULL | ค่าจริง ความถี่ และบริบท state/country/postal_code | เป็นค่าที่มีความหมายจริงหรือ missing token ของต้นทาง ห้ามลบแถวเพียงเพราะตรงรายการคำเตือน |
| city ของ event กับ user ต่างกัน | ตาราง comparison แสดงจำนวนแยกตามกรณี | ที่ตั้งผู้ใช้กับสถานที่เกิด event เป็นคนละบริบท จึงไม่ใช้แทนกันอัตโนมัติ |
| shipped_at ก่อน created_at ของ item | วันที่จากทั้ง item และ order พร้อมระยะห่าง | created_at แต่ละตารางหมายถึงอะไร และควรใช้วันใดคำนวณระยะเวลาจัดส่ง |

หลังยืนยันแล้วจึงบันทึกกฎและเหตุผลลง Data Dictionary ก่อนสร้าง cleaned/curated ส่วนนี้ไม่เติม city ไม่เปลี่ยนวันที่ และไม่ตัดแถวออกจากตารางต้นฉบับ

## ตรวจบริบทและความหมายของข้อความ null ใน city

รันหลังโหลด CSV ส่วนนี้รวบรวมหลักฐานจากไฟล์ต้นทางและพื้นที่ใกล้เคียงในข้อมูล ไม่แก้ไขค่า city และไม่สามารถยืนยันว่าเป็นชื่อเมืองจริงหรือ missing value จาก CSV เพียงอย่างเดียว การยืนยันขั้นสุดท้ายต้องใช้คำอธิบายจากแหล่งข้อมูลหรือฐานข้อมูลสถานที่ที่เชื่อถือได้

แยก **ข้อความ `null`**, ค่าว่างจริง และข้อความคล้าย missing อื่น ๆ ออกจากกัน ชื่อเมืองอื่นที่พบในพื้นที่เดียวกันเป็นเพียง candidate ไม่ใช่ค่าทดแทนที่ยืนยันแล้ว

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown

if "users" not in globals() or "events" not in globals():
    raise RuntimeError("รันเซลล์โหลดข้อมูล users และ events ก่อน")

CITY_NULL_WORDS = {"null", "na", "n/a", "none", "nan"}

def city_normalize(series):
    return series.astype("string").str.strip().str.casefold()

def city_is_null_text(series):
    return city_normalize(series).eq("null").fillna(False)

def city_is_candidate(series):
    text = city_normalize(series)
    return text.notna() & text.ne("").fillna(False) & ~text.isin(CITY_NULL_WORDS)

# อ่านเฉพาะ city จากไฟล์ดิบ เพื่อไม่ให้ pandas แปลงคำว่า null เป็น NA
city_raw_summary = []
city_raw_null_values = {}
for name, df in [("users", users), ("events", events)]:
    path = DATASET_DIR / f"{name}.csv"
    raw_counts = {"total_rows": 0, "empty_text_rows": 0, "whitespace_only_rows": 0,
                  "literal_null_rows": 0, "other_null_like_rows": 0}
    token_counts = pd.Series(dtype="int64")
    for chunk in pd.read_csv(path, usecols=["city"], dtype="string", encoding="utf-8-sig",
                             keep_default_na=False, na_filter=False, chunksize=250_000):
        text = chunk["city"]
        norm = city_normalize(text)
        is_null = norm.eq("null")
        raw_counts["total_rows"] += len(chunk)
        raw_counts["empty_text_rows"] += int(text.eq("").sum())
        raw_counts["whitespace_only_rows"] += int((text.ne("") & norm.eq("")).sum())
        raw_counts["literal_null_rows"] += int(is_null.sum())
        raw_counts["other_null_like_rows"] += int(norm.isin(CITY_NULL_WORDS - {"null"}).sum())
        token_counts = token_counts.add(text.loc[is_null].map(repr).value_counts(), fill_value=0)
    city_raw_null_values[name] = token_counts.astype("int64").rename_axis("raw_city_repr").reset_index(name="rows")
    city_raw_summary.append({"table": name, **raw_counts,
                             "loaded_literal_null_rows": int(city_is_null_text(df["city"]).sum()),
                             "loaded_actual_na_rows": int(df["city"].isna().sum())})

city_raw_summary = pd.DataFrame(city_raw_summary)
display(city_raw_summary)
for name, counts in city_raw_null_values.items():
    display(Markdown(f"#### {name}: รูปแบบข้อความ null ใน CSV จริง"))
    display(counts)

### 1. null อยู่ในพื้นที่ใด และมีชื่อเมืองอื่นในพื้นที่เดียวกันหรือไม่

users ใช้ country + state + postal_code ส่วน events ไม่มี country จึงใช้ state + postal_code ภายใน events เท่านั้น คีย์ที่ว่าง/คล้าย missing ไม่นำไปจับคู่ ไม่เชื่อมด้วย postal_code อย่างเดียวเพราะรหัสไปรษณีย์ไม่จำเป็นต้องไม่ซ้ำข้ามประเทศ

หนึ่งรหัสไปรษณีย์อาจครอบคลุมหลายเมืองหรือมีชื่อสะกดต่างกัน จำนวน candidate มากกว่าหนึ่งจึงไม่ได้พิสูจน์ว่าข้อมูลผิด และแม้พบชื่อเดียวก็ยังไม่ควรแทนค่าอัตโนมัติ

In [ ]:
def city_area_evidence(df, keys):
    work = df[keys + ["city"]].copy()
    for key in keys:
        work[key] = city_normalize(work[key])
    work["city_normalized"] = city_normalize(work["city"])
    valid_key = pd.Series(True, index=work.index)
    for key in keys:
        valid_key &= work[key].notna() & work[key].ne("").fillna(False) & ~work[key].isin(CITY_NULL_WORDS)
    null_mask = city_is_null_text(work["city"])
    groups = (work.loc[null_mask & valid_key].groupby(keys, dropna=False)
              .size().rename("null_rows").reset_index())
    eligible = work.loc[valid_key & city_is_candidate(work["city"])]
    # จำกัดข้อมูลชื่อเมืองให้เหลือเฉพาะพื้นที่ที่มี null ก่อน aggregate
    matches = eligible.merge(groups[keys], on=keys, how="inner", validate="many_to_one")
    candidates = (matches.groupby(keys + ["city_normalized"], dropna=False)
                  .size().rename("candidate_rows").reset_index())
    candidate_counts = candidates.groupby(keys, dropna=False).size().rename("candidate_city_count").reset_index()
    evidence = groups.merge(candidate_counts, on=keys, how="left", validate="one_to_one")
    evidence["candidate_city_count"] = evidence["candidate_city_count"].fillna(0).astype(int)
    evidence["evidence"] = evidence["candidate_city_count"].map(
        lambda n: "no_candidate_in_dataset" if n == 0 else "one_candidate_not_verified" if n == 1 else "multiple_candidates")
    return evidence.sort_values("null_rows", ascending=False), candidates.sort_values("candidate_rows", ascending=False), int((null_mask & ~valid_key).sum())

city_area_reports = {}
city_candidate_reports = {}
city_unmatched_key_counts = {}
for name, df, keys in [("users", users, ["country", "state", "postal_code"]),
                       ("events", events, ["state", "postal_code"])]:
    evidence, candidates, excluded = city_area_evidence(df, keys)
    city_area_reports[name] = evidence
    city_candidate_reports[name] = candidates
    city_unmatched_key_counts[name] = excluded
    display(Markdown(f"#### {name}"))
    print(f"แถว null ที่เทียบพื้นที่ไม่ได้เพราะคีย์ไม่ครบ/คล้าย missing: {excluded:,}")
    display(evidence.groupby("evidence")["null_rows"].sum().rename("null_rows").reset_index())
    print("พื้นที่ที่มี null มากที่สุด 20 กลุ่ม (ผลเต็มอยู่ใน city_area_reports)")
    display(evidence.head(20))
    print("ชื่อเมือง candidate ในพื้นที่เหล่านั้น 20 แถวแรก (ไม่ใช่ค่าที่รับรองให้ใช้แทน)")
    display(candidates.head(20))

### 2. ตรวจพิกัดของ users ที่ city เป็น null

events ไม่มี latitude/longitude จึงตรวจส่วนนี้เฉพาะ users และไม่เอาพิกัดของผู้ใช้ไปถือว่าเป็นตำแหน่งของ event การจับคู่ใช้ country + state + พิกัดปัด 5 ตำแหน่งเพื่อสำรวจเท่านั้น ไม่ใช่ geocoding และไม่รับรองว่าผู้ใช้ทั้งหมด ณ พิกัดนั้นอยู่เมืองเดียวกัน

In [ ]:
user_geo_context = users[["id", "city", "country", "state", "postal_code", "latitude", "longitude"]].copy()
for column in ["latitude", "longitude"]:
    numeric = pd.to_numeric(user_geo_context[column], errors="coerce")
    limit = 90 if column == "latitude" else 180
    user_geo_context[column + "_5dp"] = numeric.where(numeric.between(-limit, limit)).round(5).astype("string")

user_geo_evidence, user_geo_candidates, user_geo_excluded = city_area_evidence(
    user_geo_context, ["country", "state", "latitude_5dp", "longitude_5dp"])
print(f"แถว null ที่เทียบพิกัดไม่ได้เพราะพิกัด/คีย์ไม่พร้อม: {user_geo_excluded:,}")
display(user_geo_evidence.groupby("evidence")["null_rows"].sum().rename("null_rows").reset_index())
display(user_geo_evidence.head(20))
display(user_geo_candidates.head(20))
print("ตัวอย่างบริบทต้นฉบับของ users ที่ city เป็น null")
display(user_geo_context.loc[city_is_null_text(user_geo_context["city"]),
                            ["id", "city", "country", "state", "postal_code", "latitude", "longitude"]].head(10))

### 3. สรุปหลักฐานและรายการที่ต้องยืนยันจากแหล่งข้อมูล

ผลในข้อมูลช่วยบอกได้ว่าคำว่า null ปรากฏกี่พื้นที่ และมีชื่อเมืองอื่นในบริบทเดียวกันหรือไม่ แต่ห้ามสรุปว่าเป็น missing value เพียงเพราะพบในหลายพื้นที่ หรือสรุปว่าเป็นชื่อเมืองจริงเพียงเพราะไม่พบ candidate

In [ ]:
city_meaning_summary_rows = []
for name in ["users", "events"]:
    evidence = city_area_reports[name]
    raw = city_raw_summary.loc[city_raw_summary["table"].eq(name)].iloc[0]
    city_meaning_summary_rows.append({
        "table": name,
        "raw_literal_null_rows": int(raw["literal_null_rows"]),
        "loaded_literal_null_rows": int(raw["loaded_literal_null_rows"]),
        "area_groups_with_null": len(evidence),
        "null_rows_with_any_candidate": int(evidence.loc[evidence["candidate_city_count"].gt(0), "null_rows"].sum()),
        "null_rows_without_candidate": int(evidence.loc[evidence["candidate_city_count"].eq(0), "null_rows"].sum()),
        "null_rows_missing_area_keys": city_unmatched_key_counts[name],
        "conclusion": "ยังยืนยันความหมายไม่ได้จาก CSV เพียงอย่างเดียว",
        "action": "เก็บค่าเดิม รอยืนยันจากต้นทาง/แหล่งอ้างอิงภูมิศาสตร์",
    })
city_meaning_summary = pd.DataFrame(city_meaning_summary_rows)
with pd.option_context("display.max_columns", None):
    display(city_meaning_summary)

# รายการพื้นที่สำหรับตรวจยืนยันต่อโดยไม่เปิดเผยชื่อ/อีเมล/ที่อยู่/IP ของผู้ใช้
city_review_queue = pd.concat([
    report.assign(source_table=name)
    for name, report in city_area_reports.items()
], ignore_index=True, sort=False)
display(city_review_queue.head(20))
print("events ไม่มี country: ค่า country ที่ว่างใน review queue หมายถึงไม่มีคอลัมน์นี้ในต้นทาง ไม่ใช่ผลตรวจประเทศ")
print("ตรวจต้นทางเพิ่มเติม: คำอธิบาย missing token, ขั้นตอน export CSV และชื่อสถานที่ตาม country/state/postal_code หรือพิกัด")

## กฎชั่วคราวรอยืนยันและติดตาม Known Issues

ใช้กฎ `provisional-v1` ตามข้อตกลงชั่วคราววันที่ 18 ตุลาคม 2026 โดยยังไม่ได้ยืนยันกับต้นทาง หลักฐานว่า city เป็น missing token เป็นเพียงสมมติฐาน ไม่ใช่การพิสูจน์

- เก็บ city เดิม สร้าง city_clean และ city_missing_token เฉพาะข้อความ `null` หลัง trim และไม่สนตัวพิมพ์ ห้ามเติมจากแหล่งอื่นหรือทิ้งแถว
- เก็บวันเวลา item เดิม เพิ่มแฟล็ก item_shipped_before_created; ถ้าวันไม่ครบ/parse ไม่ได้/ไม่มี timezone ให้แฟล็กเป็น NA และตรวจด้วยกฎวันที่เดิม ไม่ถือว่าเป็น False
- ส่วนนี้สร้างสำเนา curated **เฉพาะ 3 ตารางที่ได้รับกฎนี้** ในหน่วยความจำ ยังไม่ใช่ curated pipeline ที่ครบทุกกฎ และยังไม่ export ไฟล์
- ข้อมูล raw และผลตรวจต้นทางเดิมยังอยู่ ใช้ dq_review_summary เป็นผลหลังรับทราบ known issue โดยไม่เปลี่ยน dq_summary ต้นฉบับ
- KNOWN_ISSUE = รับทราบและรอยืนยัน ไม่ใช่ PASS ถ้าจำนวน/ตัวหาร/จำนวนแถว/hash ไฟล์ต่างจาก baseline ให้ WARN ใหม่ แม้จำนวนลดลงหรือเป็นศูนย์
- รันหลังส่วนตรวจคุณภาพเพื่อให้มี dq_summary ของรอบปัจจุบัน หากเปลี่ยน DataFrame ต้องรันการตรวจใหม่ด้วย

In [ ]:
import hashlib
import json
from pathlib import Path
from IPython.display import display

baseline_path = DATASET_DIR.parent / "code" / "dq_known_issues_baseline.json"
known_baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
if "dq_summary" not in globals():
    raise RuntimeError("รันส่วนตรวจคุณภาพและเซลล์สรุป dq_summary ของรอบปัจจุบันก่อน")

def add_provisional_city_columns(df):
    result = df.copy(deep=True)
    flag = df["city"].astype("string").str.strip().str.casefold().eq("null").fillna(False)
    result["city_missing_token"] = flag.astype(bool)
    # ไม่ trim หรือเปลี่ยนรูปแบบเมืองอื่น เก็บข้อความต้นฉบับใน city เสมอ
    result["city_clean"] = df["city"].astype("string").mask(flag, pd.NA)
    return result

def provisional_item_date_flags(df):
    dates = {}
    for column in ["created_at", "shipped_at"]:
        text = df[column].astype("string").str.strip()
        has_zone = text.str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", regex=True, case=False, na=False)
        dates[column] = pd.to_datetime(text.where(has_zone), format="mixed", utc=True, errors="coerce")
    comparable = dates["created_at"].notna() & dates["shipped_at"].notna()
    flags = dates["shipped_at"].lt(dates["created_at"]).astype("boolean").where(comparable, pd.NA)
    return flags, comparable

users_curated = add_provisional_city_columns(users)
events_curated = add_provisional_city_columns(events)
order_items_curated = order_items.copy(deep=True)
item_date_flags, item_date_comparable = provisional_item_date_flags(order_items)
order_items_curated["item_shipped_before_created"] = item_date_flags
order_items_curated["item_dates_comparable"] = item_date_comparable

curated_dataframes = {"users": users_curated, "events": events_curated, "order_items": order_items_curated}
display(pd.DataFrame([
    {"table": name, "raw_rows": len(dataframes[name]), "curated_rows": len(df),
     "added_columns": ", ".join(c for c in df.columns if c not in dataframes[name].columns)}
    for name, df in curated_dataframes.items()
]))

### ติดตาม Known Issues ของ snapshot เดิม

Baseline ถูกเก็บเป็นไฟล์แยก ไม่สร้างใหม่จากค่าที่พบทุกครั้ง จึงตรวจการเปลี่ยนต้นทางได้ ห้ามแก้ baseline เพียงเพื่อให้ WARN หาย ต้องทบทวนข้อมูลชุดใหม่และบันทึกเหตุผลก่อน ตัวเลข 30.33% ของวันที่ใช้ตัวหาร 118,281 แถวที่มีวันทั้งคู่และ timezone ถูกต้อง ไม่ใช่จำนวน order_items ทั้งหมด

In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def known_issue_status(actual, expected, comparable, expected_comparable, snapshot_matches):
    return ("KNOWN_ISSUE" if snapshot_matches and actual == expected and comparable == expected_comparable
            else "WARN")

observed_known = {
    "CITY-01-users": (int(users_curated["city_missing_token"].sum()), len(users)),
    "CITY-01-events": (int(events_curated["city_missing_token"].sum()), len(events)),
    "DATE-01-items": (int(item_date_flags.sum()), int(item_date_comparable.sum())),
}
snapshot_matches = {
    name: len(dataframes[name]) == info["rows"] and file_sha256(DATASET_DIR / f"{name}.csv") == info["sha256"]
    for name, info in known_baseline["tables"].items()
}
review_rows = []
dq_review_summary = dq_summary.copy(deep=True)
for issue in known_baseline["issues"]:
    actual, comparable = observed_known[issue["id"]]
    status = known_issue_status(actual, issue["expected_count"], comparable,
                               issue["expected_checked_rows"], snapshot_matches[issue["table"]])
    mask = dq_review_summary["table"].eq(issue["table"]) & dq_review_summary["rule"].eq(issue["rule"])
    dq_row = dq_review_summary.loc[mask]
    # ไม่กลบคำเตือนอื่นที่รวมอยู่ในกฎ null_like_text เช่น N/A และไม่กลบ FAIL/SKIP
    dq_consistent = (len(dq_row) == 1 and dq_row.iloc[0]["status"] in ["PASS", "WARN"]
                     and dq_row.iloc[0]["issue_rows"] == actual
                     and dq_row.iloc[0]["checked_rows"] == comparable)
    if not dq_consistent:
        status = "WARN"
    review_rows.append({
        "issue_id": issue["id"], "table": issue["table"], "status": status,
        "expected_rows": issue["expected_count"], "actual_rows": actual,
        "expected_checked_rows": issue["expected_checked_rows"], "actual_checked_rows": comparable,
        "snapshot_matches": snapshot_matches[issue["table"]], "dq_result_matches": dq_consistent,
        "source_confirmation": "pending", "rule_version": known_baseline["version"],
    })
    if len(dq_row) == 1 and dq_row.iloc[0]["status"] in ["PASS", "WARN"]:
        dq_review_summary.loc[mask, "status"] = status
        dq_review_summary.loc[mask, "note"] = (
            f"{issue['id']}; provisional-v1; รอยืนยันต้นทาง; "
            + ("ตรง baseline เดิม" if status == "KNOWN_ISSUE" else "ต้องทบทวน baseline/ผล DQ/ข้อมูลที่เปลี่ยน")
        )

known_issue_review = pd.DataFrame(review_rows)
display(known_issue_review)
display(pd.crosstab(dq_review_summary["table"], dq_review_summary["status"]).reindex(
    columns=["FAIL", "WARN", "SKIP", "KNOWN_ISSUE", "PASS"], fill_value=0))
display(dq_review_summary.loc[~dq_review_summary["status"].isin(["PASS", "KNOWN_ISSUE"])])
print("KNOWN_ISSUE ไม่ใช่ PASS และยังรอการยืนยันจากต้นทาง ดู known_issue_review ควบคู่กับ dq_review_summary")

### การใช้ข้อมูลและเงื่อนไขย้อนกลับกฎ

- รายงานเมืองใช้ city_clean และแสดง NULL เป็น Unknown ที่ชั้นรายงาน โดยยังตรวจย้อนกลับ city ได้ ห้ามนำ Unknown ไปเขียนทับ city
- ยังไม่ใช้ order_items.created_at คำนวณระยะเวลาจนกว่าจะยืนยันกับต้นทาง แม้ทราบคำอธิบายว่าเป็นเวลาสร้าง order item แล้ว แต่สาเหตุที่อยู่หลัง shipped_at ยังไม่ยืนยัน
- ถ้ารายงานต้องใช้ระยะเวลา: orders.created_at → orders.shipped_at คือเวลาจนส่งออก; orders.created_at → orders.delivered_at คือเวลาจนส่งถึง; shipped_at → delivered_at คือเวลาขนส่ง ทั้งหมดต้องตรวจคู่วันที่และความหมายก่อนใช้ ไม่จำเป็นต้องสร้าง metric เหล่านี้ในงาน cleaning
- ถ้าต้นทางยืนยันว่า null เป็นชื่อสถานที่จริง ให้ยกเลิก CITY-01 แล้วสร้าง city_clean ใหม่จาก city เดิม โดยบันทึกเวอร์ชันกฎใหม่ ห้ามเขียนทับหลักฐาน baseline เดิมโดยไม่บันทึกการเปลี่ยน
- คำถามที่ยังต้องส่งให้เจ้าของข้อมูล: null ใน city หมายถึงอะไร และเหตุใด order_items.created_at จึงอยู่หลัง shipped_at ในบางรายการ ยังไม่ได้ส่งคำถามออกนอกโปรเจกต์

## สร้าง curated ครบ 7 ตาราง และตรวจเทียบก่อน–หลัง

ส่วนนี้ใช้ curated_schema.json และ curated_pipeline.py อ่าน CSV ต้นฉบับเป็นข้อความทีละ chunk เพื่อรักษาความแม่นยำก่อนแปลงราคาเป็น Decimal(18,6) แบบ ROUND_HALF_UP และวันที่เป็น UTC ความละเอียด microsecond

รันส่วนนี้ได้โดยไม่โหลดทุกตารางเข้า RAM ก่อน รันแต่ละครั้งสร้าง batch ใหม่ใน outputs/curated ไม่เขียนทับ raw หรือ batch เดิม กฎ city/วันที่ยังเป็น provisional-v1 รอยืนยันต้นทาง ถ้าเกิดข้อผิดพลาดหรือ baseline เปลี่ยนจะหยุดเพื่อทบทวน และเก็บรายงานไว้ใน batch นั้น

ไฟล์ Parquet ครบทั้ง 7 อยู่ใน curated_files ซึ่งแยกจาก curated_dataframes แบบชั่วคราว 3 ตารางในส่วนก่อนหน้า ยังไม่อัปโหลด AWS

In [ ]:
from pathlib import Path
import sys
import importlib
import pandas as pd
from IPython.display import display

PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "code" / "curated_pipeline.py").is_file())
CODE_DIR = PROJECT_DIR / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# โปรเจกต์นี้ติดตั้ง pyarrow ไว้ใน .runtime_dependencies
# ถ้าใช้ Python environment อื่น ให้ติดตั้ง pandas และ pyarrow ใน environment นั้น
import curated_pipeline
importlib.reload(curated_pipeline)

# ใช้ batch_id ตามต้องการ (ไม่ใส่ = auto timestamp)
curated_output_dir, curated_validation = curated_pipeline.run_curated(PROJECT_DIR, batch_id='my-baseline-v1')
curated_files = {name: curated_output_dir / f"{name}.parquet"
                 for name in curated_validation["tables"]}
print("สถานะ:", curated_validation["status"])
print("โฟลเดอร์ผลลัพธ์:", curated_output_dir)

In [ ]:
curated_before_after = pd.DataFrame([
    {"table": name, "raw_rows": info["raw_rows"],
     "curated_rows": info["curated_rows"], "quarantined_rows": info["quarantined_rows"]}
    for name, info in curated_validation["tables"].items()
])
display(curated_before_after)
display(pd.DataFrame(curated_validation["known_issues"]))

curated_checks = pd.DataFrame(curated_validation["checks"])
display(curated_checks)

curated_money_reconciliation = pd.DataFrame([
    {"table": name, "column": column, **metrics}
    for name, info in curated_validation["tables"].items()
    for column, metrics in info["money"].items()
])
with pd.option_context("display.max_columns", None):
    display(curated_money_reconciliation)
print("ยอดเงินแสดง Decimal ตามจริง ผลต่างจาก rounding แยกไว้ ไม่ใช่ยอดเงินหายจากการลบแถว")

In [ ]:
import pyarrow.parquet as pq

curated_column_comparison = pd.DataFrame([
    {"table": name, "column": column, **metrics}
    for name, info in curated_validation["tables"].items()
    for column, metrics in info["columns"].items()
])
display(curated_column_comparison)

for name, file in curated_files.items():
    print(f"{name}: {pq.ParquetFile(file).schema_arrow}")

def read_curated(table_name, columns=None):
    # อ่านทีละตาราง/เลือกคอลัมน์เพื่อลด RAM โดยเฉพาะ events
    return pq.read_table(curated_files[table_name], columns=columns).to_pandas()

display(read_curated("users", ["id", "city", "city_clean", "city_missing_token"]).head())
display(read_curated("order_items", ["id", "created_at", "shipped_at", "item_shipped_before_created"]).head())